# Sparse Merkle tree explainer

Olympus uses a **256-level sparse Merkle tree** for ledger state. This notebook shows the
real API (`SparseMerkleTree`, `prove_existence`, `prove_nonexistence`) and then shrinks the
idea to a toy 4-bit tree so the proof path can be seen directly.


In [ ]:
from protocol.hashes import hash_bytes, record_key
from protocol.ssmf import SparseMerkleTree, verify_nonexistence_proof, verify_proof


In [1]:
tree = SparseMerkleTree()
records = [
    ('document', 'budget-2026', 1),
    ('document', 'audit-2024', 1),
    ('document', 'memo-7', 1),
]
for record_type, record_id, version in records:
    key = record_key(record_type, record_id, version)
    value_hash = hash_bytes(f'{record_type}:{record_id}:v{version}'.encode())
    tree.update(key, value_hash, 'docling@2.3.1', 'v1')

existing_key = record_key('document', 'budget-2026', 1)
missing_key = record_key('document', 'missing', 1)
existence = tree.prove_existence(existing_key)
missing = tree.prove_nonexistence(missing_key)

print(f'Root hash: {tree.get_root().hex()}')
print(f'Existing key prefix: {existence.key.hex()[:16]}...')
print(f'Missing key prefix: {missing.key.hex()[:16]}...')
print(f'Existence proof valid: {verify_proof(existence)}')
print(f'Non-existence proof valid: {verify_nonexistence_proof(missing)}')
print(f'Existence sibling count: {len(existence.siblings)}')
print(f'Non-existence sibling count: {len(missing.siblings)}')


Root hash: 1201324ae0c3cc652a321900625c1ef3691e98b9cffcccb63c996cfe1f2e8ee8
Existing key prefix: 6a6289bd1cc10558...
Missing key prefix: 91d1f613b0525473...
Existence proof valid: True
Non-existence proof valid: True
Existence sibling count: 256
Non-existence sibling count: 256


In [2]:
            toy_tree = r"""root
├── 0***  empty subtree
└── 1***
    ├── 10**
    │   ├── 100*  empty subtree
    │   └── 101*
    │       ├── 1010  target leaf: budget-2026
    │       └── 1011  sibling leaf
    └── 11**  empty subtree

Inclusion proof path for 1010
  level 3 sibling: 1011
  level 2 sibling: 100*
  level 1 sibling: 11**
  level 0 sibling: 0***"""
            print(toy_tree)


root
├── 0***  empty subtree
└── 1***
    ├── 10**
    │   ├── 100*  empty subtree
    │   └── 101*
    │       ├── 1010  target leaf: budget-2026
    │       └── 1011  sibling leaf
    └── 11**  empty subtree

Inclusion proof path for 1010
  level 3 sibling: 1011
  level 2 sibling: 100*
  level 1 sibling: 11**
  level 0 sibling: 0***


In the real 256-level tree the proof contains **256 sibling hashes**, one for every bit of the
32-byte key. The toy tree above is only there to make the branching intuitive: each proof says,
“here is the sibling you must combine with the current hash at each level until you reconstruct
the committed root.”
